<h1 style="font-size:40px;"> MLlib: Machine Learning con Spark </h1>

In [1]:
import numpy as np
import pyspark.sql.functions as F
from pyspark import SparkConf
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.feature import VectorAssembler, VectorIndexer
from pyspark.ml.regression import DecisionTreeRegressor, RandomForestRegressor
from pyspark.sql import SparkSession

In [2]:
conf = (
    SparkConf()
    .setAppName("[ICAI] ML a fondo")
    .set("spark.executor.memory", "7g")
    .set("spark.executor.cores", "5")
    .set("spark.default.parallelism", 800)
    .set("spark.sql.shuffle.partitions", 800)
)

In [3]:
spark = SparkSession.builder.config(conf=conf).enableHiveSupport().getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [4]:
spark.sparkContext.setLogLevel("FATAL")

# Tuning de modelos

![](img/rf.jpg)

Como hemos visto en el ejercicio anterior estabamos consiguiendo peores resultados en el [Random Forest](https://es.wikipedia.org/wiki/Random_forest) que un solo árbol, veamos como solucionar esto:

In [5]:
trainDF, testDF = (
    spark.read.options(header=True, inferSchema=True)
    .csv("/datos/hour.csv")
    .drop("casual", "registerd", "instant", "dteday")
    .randomSplit([0.7, 0.3], seed=1234)
)

In [6]:
trainDF.cache()
testDF.cache()
print("## Número de registros en `trainDF`: {}".format(trainDF.count()))
print("## Número de registros en `testDF`: {}".format(testDF.count()))

## Número de registros en `trainDF`: 12201
## Número de registros en `testDF`: 5178


In [7]:
featuresCols = trainDF.columns[:-1]
featuresCols

['season',
 'yr',
 'mnth',
 'hr',
 'holiday',
 'weekday',
 'workingday',
 'weathersit',
 'temp',
 'atemp',
 'hum',
 'windspeed',
 'registered']

#### Primero montamos de nuevo el árbol:

In [8]:
vectorAssembler = VectorAssembler(inputCols=featuresCols, outputCol="rawFeatures")

In [9]:
vectorIndexer = VectorIndexer(inputCol="rawFeatures", outputCol="features")

In [10]:
dt = DecisionTreeRegressor(labelCol="cnt")

In [11]:
pipeline = Pipeline(stages=[vectorAssembler, vectorIndexer, dt])

In [12]:
model = pipeline.fit(trainDF)

In [13]:
evaluator = RegressionEvaluator(labelCol="cnt")

In [14]:
rmse_train = evaluator.evaluate(model.transform(trainDF))
rmse_valid = evaluator.evaluate(model.transform(testDF))

In [15]:
print("## RMSE (Train): {:.3f}".format(rmse_train))
print("## RMSE (Valid): {:.3f}".format(rmse_valid))

## RMSE (Train): 32.151
## RMSE (Valid): 33.357


#### Ahora el bosque:

In [16]:
rf = RandomForestRegressor(labelCol="cnt")

In [17]:
pipeline2 = Pipeline(stages=model.stages[:-1] + [rf])

In [18]:
model2 = pipeline2.fit(trainDF)

In [19]:
rmse_train = evaluator.evaluate(model2.transform(trainDF))
rmse_valid = evaluator.evaluate(model2.transform(testDF))

In [20]:
print("## RMSE (Train): {:.3f}".format(rmse_train))
print("## RMSE (Valid): {:.3f}".format(rmse_valid))

## RMSE (Train): 42.501
## RMSE (Valid): 42.884


Los modelos más complejos tienen varios hiper-parámetros que hay que configurar para conseguir la mayor *performance*, a esta búsqueda de la configuración óptima se le suele conocer como [*tuning*](https://en.wikipedia.org/wiki/Hyperparameter_optimization). Veamos los parámetros del modelo en cuestión:

In [21]:
print(RandomForestRegressor().explainParams())

bootstrap: Whether bootstrap samples are used when building trees. (default: True)
cacheNodeIds: If false, the algorithm will pass trees to executors to match instances with nodes. If true, the algorithm will cache node IDs for each instance. Caching can speed up training of deeper trees. Users can set how often should the cache be checkpointed or disable it by setting checkpointInterval. (default: False)
checkpointInterval: set checkpoint interval (>= 1) or disable checkpoint (-1). E.g. 10 means that the cache will get checkpointed every 10 iterations. Note: this setting will be ignored if the checkpoint directory is not set in the SparkContext. (default: 10)
featureSubsetStrategy: The number of features to consider for splits at each tree node. Supported options: 'auto' (choose automatically for task: If numTrees == 1, set to 'all'. If numTrees > 1 (forest), set to 'sqrt' for classification and to 'onethird' for regression), 'all' (use all features), 'onethird' (use 1/3 of the featur

In [22]:
rf3 = RandomForestRegressor(labelCol="cnt", numTrees=200, maxDepth=10)

In [23]:
model3 = Pipeline(stages=model.stages[:-1] + [rf3]).fit(trainDF)

In [24]:
rmse_train = evaluator.evaluate(model3.transform(trainDF))
rmse_valid = evaluator.evaluate(model3.transform(testDF))

In [25]:
print("## RMSE (Train): {:.3f}".format(rmse_train))
print("## RMSE (Valid): {:.3f}".format(rmse_valid))

## RMSE (Train): 17.991
## RMSE (Valid): 21.579


¡Ya hemos conseguido mejores resultados!

### Cross-validation y búsqueda por malla

![](img/cv.png)

En Spark MLlib existen funciones para hacer fácil la búsqueda de hiperparámetros y la [validación cruzada](https://en.wikipedia.org/wiki/Cross-validation_(statistics))


In [26]:
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

In [27]:
paramGrid = (
    ParamGridBuilder().addGrid(rf.numTrees, [100, 200, 300]).addGrid(rf.maxDepth, [4, 10]).build()
)

In [28]:
crossval = CrossValidator(
    estimator=pipeline2, estimatorParamMaps=paramGrid, evaluator=evaluator, numFolds=3
)

**CUIDADO** Este `fit` puede durar varios minutos:

In [29]:
cvModel = crossval.fit(trainDF)

In [30]:
cvModel.avgMetrics

[52.665875244066655,
 22.984465213175763,
 52.61428835803573,
 22.96541831829431,
 51.55209794322068,
 22.83834880137572]

In [31]:
mejor = np.argsort(cvModel.avgMetrics)[0]

In [32]:
mejor

5

In [33]:
cvModel.avgMetrics[mejor]

22.83834880137572

In [34]:
cvModel.getEstimatorParamMaps()[mejor]

{Param(parent='RandomForestRegressor_d22c4e96536d', name='numTrees', doc='Number of trees to train (>= 1).'): 300,
 Param(parent='RandomForestRegressor_d22c4e96536d', name='maxDepth', doc='Maximum depth of the tree. (>= 0) E.g., depth 0 means 1 leaf node; depth 1 means 1 internal node + 2 leaf nodes. Must be in range [0, 30].'): 10}

In [35]:
rmse_train = evaluator.evaluate(cvModel.transform(trainDF))
rmse_valid = evaluator.evaluate(cvModel.transform(testDF))

In [36]:
print("## RMSE (Train): {:.3f}".format(rmse_train))
print("## RMSE (Valid): {:.3f}".format(rmse_valid))

## RMSE (Train): 17.801
## RMSE (Valid): 21.457


# Clustering: K-Means

![](img/wikimedia.png)

Veamos un ejemplo de modelo no supervisado. Usaremos para ello un dataset de artículos de wikipedia que se puede encontrar en: https://dumps.wikimedia.org/

In [37]:
wiki_df = spark.read.parquet("/datos/wiki.parquet").repartition(800).cache()

In [38]:
wiki_df.count()

111495

In [39]:
wiki_df.printSchema()

root
 |-- id: long (nullable = true)
 |-- title: string (nullable = true)
 |-- lastrev_pdt_time: timestamp (nullable = true)
 |-- revid: long (nullable = true)
 |-- comment: string (nullable = true)
 |-- contributorid: long (nullable = true)
 |-- contributorusername: string (nullable = true)
 |-- contributorip: string (nullable = true)
 |-- text: string (nullable = true)



In [40]:
wiki_df.limit(10).toPandas()

/opt/share/anaconda/lib/python3.11/site-packages/pyspark/sql/pandas/conversion.py:251: FutureWarning: Passing unit-less datetime64 dtype to .astype is deprecated and will raise in a future version. Pass 'datetime64[ns]' instead
  series = series.astype(t, copy=False)


,id,title,lastrev_pdt_time,revid,comment,contributorid,contributorusername,contributorip,text
0,7523301,Syrian Premier League,2016-03-04 16:29:02,708327892,Removed image per [[Wikipedia:Files for discus...,4842600.0,Explicit,None,{{Infobox football league\n| logo = \n| ...
1,503927,Parachutes (album),2016-03-03 09:33:52,708097430,/* Critical reception */ Replacing user review,4279454.0,Holiday56,None,{{EngvarB|date=September 2013}}\n{{Use dmy dat...
2,12418406,Vinous-breasted sparrowhawk,2016-03-04 16:52:23,708330760,Accipitriformes-stub,730134.0,Pelagic,None,{{Taxobox\n| name = Vinous-breasted sparrowhaw...
3,41099217,"Newcastle upon Tyne West by-election, 1940",2016-03-04 15:26:30,708319709,Whoever wrote this originally should be shot,13249188.0,Marplesmustgo,None,The '''[[Newcastle upon Tyne West (UK Parliame...
4,6024369,Tim Stoddard,2016-03-04 11:07:24,708281097,migrating [[Wikipedia:Persondata|Persondata]] ...,24420788.0,KasparBot,None,{{BLP sources|date=January 2008}}\n{{Infobox b...
5,8040205,"O come, O come, Emmanuel",2016-03-03 01:06:52,708044902,"/* Text */ Restored ""are essentially the same ...",3784662.0,Liberalartist,None,"'''''O come, O come, Emmanuel''''' is a [[Chri..."
6,5870652,Sal Butera,2016-03-03 19:33:26,708177575,migrating [[Wikipedia:Persondata|Persondata]] ...,24420788.0,KasparBot,None,{{Use mdy dates|date=July 2014}}\n{{Infobox ba...
7,6189968,Jake Gibbs,2016-03-05 05:05:27,708407091,migrating [[Wikipedia:Persondata|Persondata]] ...,24420788.0,KasparBot,None,{{Infobox baseball biography\n|name=Jake Gibbs...
8,16341,John Paul Jones,2016-03-04 15:24:45,708319501,Reverted edits by [[Special:Contributions/205....,25523690.0,CAPTAIN RAJU,None,{{about||the Led Zeppelin musician|John Paul J...
9,136920,"Medford, New Jersey",2016-03-03 12:19:10,708119678,There are two area codes,NaN,None,2600:1002:B122:D438:7DB3:73D3:A6F9:B52A,{{Infobox settlement\n| name ...


Empezamos por un tratemiento del texto como ya hemos visto:

In [41]:
for i in wiki_df.select("text").take(4):
    print(i.text)
    print("------------\n\n")

{{Infobox football league
| logo       = 
| pixels     = 
| country    = Syria
| confed     = AFC
| founded    = 1966
| teams      = 20
| relegation = 2. Syrian League
| levels     = 1
| domest_cup = [[Syrian Cup]]<br />[[Syrian Super Cup]]
| confed_cup = [[AFC Cup]]<br />[[Arab Champions League]]
| champions  = [[Al-Jaish SC (Damascus)|Al-Jaish]]
| season     =  2014???15
| most successful club = [[Al-Jaish SC (Damascus)|Al-Jaish]] (13 titles) 
| website    = 
| current    = [[2015???16 Syrian Premier League]]
}}

The '''Syrian League''' is the highest division in [[Association football|football]] in [[Syria]]. The league began to play in 1966.

==Current clubs (2015???16)==
===Group 1===
*[[Al-Hurriya SC|Al-Hurriya]] (Aleppo)
*[[Al-Jaish SC (Damascus)|Al-Jaish]] (Damascus)
*[[Al-Jazeera SC Hasakah|Al-Jazeera]]  (Hasakah)
*[[Al-Karamah SC|Al-Karamah]] (Homs)
*[[Al-Majd SC|Al-Majd]] (Damascus)
*[[Al-Muhafaza SC|Al-Muhafaza]] (Damascus)
*[[Taliya SC|Al-Taliya]] (Hama)
*[[Hutteen SC|Hutt

In [42]:
from pyspark.ml.feature import (
    IDF,
    HashingTF,
    Normalizer,
    RegexTokenizer,
    StopWordsRemover,
)

##### Step 1: Natural Language Processing: RegexTokenizer: Convert the lowerText col to a bag of words

In [43]:
tokenizer = RegexTokenizer().setInputCol("text").setOutputCol("words").setPattern(r"\W+")

In [44]:
wiki_words_df = tokenizer.transform(wiki_df)

In [45]:
wiki_words_df.printSchema()

root
 |-- id: long (nullable = true)
 |-- title: string (nullable = true)
 |-- lastrev_pdt_time: timestamp (nullable = true)
 |-- revid: long (nullable = true)
 |-- comment: string (nullable = true)
 |-- contributorid: long (nullable = true)
 |-- contributorusername: string (nullable = true)
 |-- contributorip: string (nullable = true)
 |-- text: string (nullable = true)
 |-- words: array (nullable = true)
 |    |-- element: string (containsNull = true)



In [46]:
np.array(wiki_words_df.select("words").first()[0])

array(['infobox', 'football', 'league', 'logo', 'pixels', 'country',
       'syria', 'confed', 'afc', 'founded', '1966', 'teams', '20',
       'relegation', '2', 'syrian', 'league', 'levels', '1', 'domest_cup',
       'syrian', 'cup', 'br', 'syrian', 'super', 'cup', 'confed_cup',
       'afc', 'cup', 'br', 'arab', 'champions', 'league', 'champions',
       'al', 'jaish', 'sc', 'damascus', 'al', 'jaish', 'season', '2014',
       '15', 'most', 'successful', 'club', 'al', 'jaish', 'sc',
       'damascus', 'al', 'jaish', '13', 'titles', 'website', 'current',
       '2015', '16', 'syrian', 'premier', 'league', 'the', 'syrian',
       'league', 'is', 'the', 'highest', 'division', 'in', 'association',
       'football', 'football', 'in', 'syria', 'the', 'league', 'began',
       'to', 'play', 'in', '1966', 'current', 'clubs', '2015', '16',
       'group', '1', 'al', 'hurriya', 'sc', 'al', 'hurriya', 'aleppo',
       'al', 'jaish', 'sc', 'damascus', 'al', 'jaish', 'damascus', 'al',
       'jaz

##### Step 2: Natural Language Processing: StopWordsRemover: Remove Stop words

In [47]:
remover = StopWordsRemover().setInputCol("words").setOutputCol("noStopWords")

In [48]:
no_stop_words_list_df = remover.transform(wiki_words_df)

In [49]:
no_stop_words_list_df.printSchema()

root
 |-- id: long (nullable = true)
 |-- title: string (nullable = true)
 |-- lastrev_pdt_time: timestamp (nullable = true)
 |-- revid: long (nullable = true)
 |-- comment: string (nullable = true)
 |-- contributorid: long (nullable = true)
 |-- contributorusername: string (nullable = true)
 |-- contributorip: string (nullable = true)
 |-- text: string (nullable = true)
 |-- words: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- noStopWords: array (nullable = true)
 |    |-- element: string (containsNull = true)



In [50]:
no_stop_words_list_df.select("id", "title", "words", "noStopWords").show(15)

+--------+--------------------+--------------------+--------------------+
|      id|               title|               words|         noStopWords|
+--------+--------------------+--------------------+--------------------+
| 7523301|Syrian Premier Le...|[infobox, footbal...|[infobox, footbal...|
|  503927|  Parachutes (album)|[engvarb, date, s...|[engvarb, date, s...|
|12418406|Vinous-breasted s...|[taxobox, name, v...|[taxobox, name, v...|
|41099217|Newcastle upon Ty...|[the, newcastle, ...|[newcastle, upon,...|
| 6024369|        Tim Stoddard|[blp, sources, da...|[blp, sources, da...|
| 8040205|O come, O come, E...|[o, come, o, come...|[o, come, o, come...|
| 5870652|          Sal Butera|[use, mdy, dates,...|[use, mdy, dates,...|
| 6189968|          Jake Gibbs|[infobox, basebal...|[infobox, basebal...|
|   16341|     John Paul Jones|[about, the, led,...|[led, zeppelin, m...|
|  136920| Medford, New Jersey|[infobox, settlem...|[infobox, settlem...|
| 5760716|Christian Birmingham|[christ

In [51]:
no_stop_words_list_df.select(F.explode("noStopWords").alias("words")).select(
    F.countDistinct("words")
).show()

+---------------------+
|count(DISTINCT words)|
+---------------------+
|              3720161|
+---------------------+



##### Step 3: HashingTF

![](img/tf-idf.png)

[*HashingTF*](https://en.wikipedia.org/wiki/Tf%E2%80%93idf) es una técnica empleada para no tener que calcular la matriz *term frequency* completa, por métodos de hashing conseguimos resultados muy cercanos con una *performance* de cálculo mucho más rápida y paralelizable.

In [52]:
hashing_tf = HashingTF().setInputCol("noStopWords").setOutputCol("hashingTF").setNumFeatures(20000)

In [53]:
featurized_data_df = hashing_tf.transform(no_stop_words_list_df)

In [54]:
featurized_data_df.printSchema()

root
 |-- id: long (nullable = true)
 |-- title: string (nullable = true)
 |-- lastrev_pdt_time: timestamp (nullable = true)
 |-- revid: long (nullable = true)
 |-- comment: string (nullable = true)
 |-- contributorid: long (nullable = true)
 |-- contributorusername: string (nullable = true)
 |-- contributorip: string (nullable = true)
 |-- text: string (nullable = true)
 |-- words: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- noStopWords: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- hashingTF: vector (nullable = true)



In [55]:
featurized_data_df.select("id", "title", "noStopWords", "hashingTF").limit(10).toPandas()

,id,title,noStopWords,hashingTF
0,7523301,Syrian Premier League,"[infobox, football, league, logo, pixels, coun...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
1,503927,Parachutes (album),"[engvarb, date, september, 2013, use, dmy, dat...","(0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, ..."
2,12418406,Vinous-breasted sparrowhawk,"[taxobox, name, vinous, breasted, sparrowhawk,...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
3,41099217,"Newcastle upon Tyne West by-election, 1940","[newcastle, upon, tyne, west, uk, parliament, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
4,6024369,Tim Stoddard,"[blp, sources, date, january, 2008, infobox, b...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
5,8040205,"O come, O come, Emmanuel","[o, come, o, come, emmanuel, christian, hymn, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
6,5870652,Sal Butera,"[use, mdy, dates, date, july, 2014, infobox, b...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
7,6189968,Jake Gibbs,"[infobox, baseball, biography, name, jake, gib...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
8,16341,John Paul Jones,"[led, zeppelin, musician, john, paul, jones, m...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
9,136920,"Medford, New Jersey","[infobox, settlement, name, medford, new, jers...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."


In [56]:
vector = featurized_data_df.select("hashingTF").first()[0]

In [57]:
type(vector)

pyspark.ml.linalg.SparseVector

In [58]:
vector.values

array([  1.,   1.,   1.,   1.,   2.,   1.,   1.,   1.,   1.,   1.,   5.,
        34.,   1.,   1.,   3.,   1.,   1.,   1.,   1.,   2.,   2.,   1.,
         1.,   1.,   1.,   1.,   2.,   1.,   1.,   5.,   7.,   1.,   1.,
         4.,   2.,   1.,   2., 110.,   3.,   1.,   1.,   1.,   2.,   2.,
         1.,   8.,   1.,   8.,   1.,   2.,   1.,   4.,   1.,   2.,   1.,
         1.,   6.,   4.,   2.,   1.,   7.,   1.,   2.,   1.,   1.,   2.,
         2.,   5.,   1.,   5.,   1.,   1.,   1.,   2.,   2.,   1.,   1.,
         2.,   6.,   1.,  20.,   8.,   1.,   1.,   1.,   1.,  16.,   5.,
         1.,   2.,   1.,   1.,   1.,  14.,   1.,   1.,   2.,   7.,  14.,
         1.,  17.,   2.,   1.,   1.,   2.,   1.,   2.,   1.,   1.,   1.,
         1.,   2.,   2.,   1.,   1.,   1.,   4.,   1.,   2.,   2.,   1.,
         1.,   2.,   1.,   1.,   1.,   1.,   1.,   3.,   1.,   1.,   1.,
         1.,   1.,   5.,   1.,   8.,   1.,   1.,   2.,   1.,  36.,   1.,
         2.,   1.,   1.,   1.,   1.,   1.,   1.,   

In [59]:
vector.toArray()

array([0., 0., 0., ..., 0., 0., 0.])

In [60]:
aux = featurized_data_df.select("id", "title", "noStopWords", "hashingTF").first()

In [61]:
aux

Row(id=7523301, title='Syrian Premier League', noStopWords=['infobox', 'football', 'league', 'logo', 'pixels', 'country', 'syria', 'confed', 'afc', 'founded', '1966', 'teams', '20', 'relegation', '2', 'syrian', 'league', 'levels', '1', 'domest_cup', 'syrian', 'cup', 'br', 'syrian', 'super', 'cup', 'confed_cup', 'afc', 'cup', 'br', 'arab', 'champions', 'league', 'champions', 'al', 'jaish', 'sc', 'damascus', 'al', 'jaish', 'season', '2014', '15', 'successful', 'club', 'al', 'jaish', 'sc', 'damascus', 'al', 'jaish', '13', 'titles', 'website', 'current', '2015', '16', 'syrian', 'premier', 'league', 'syrian', 'league', 'highest', 'division', 'association', 'football', 'football', 'syria', 'league', 'began', 'play', '1966', 'current', 'clubs', '2015', '16', 'group', '1', 'al', 'hurriya', 'sc', 'al', 'hurriya', 'aleppo', 'al', 'jaish', 'sc', 'damascus', 'al', 'jaish', 'damascus', 'al', 'jazeera', 'sc', 'hasakah', 'al', 'jazeera', 'hasakah', 'al', 'karamah', 'sc', 'al', 'karamah', 'homs', 'al'

##### Step 4: IDF

Calculamos ahora los puntuaciones inversas

In [62]:
idf = IDF().setInputCol("hashingTF").setOutputCol("idf")
idf_model = idf.fit(featurized_data_df)

In [63]:
idf_model

IDFModel: uid=IDF_6b6599bb6dd8, numDocs=111495, numFeatures=20000

In [64]:
idf_model.transform(featurized_data_df).select("text", "hashingTF", "idf").limit(10).toPandas()

,text,hashingTF,idf
0,{{Infobox football league\n| logo = \n| ...,"(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
1,{{EngvarB|date=September 2013}}\n{{Use dmy dat...,"(0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 3.8702686767778887, 0.0, ..."
2,{{Taxobox\n| name = Vinous-breasted sparrowhaw...,"(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
3,The '''[[Newcastle upon Tyne West (UK Parliame...,"(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
4,{{BLP sources|date=January 2008}}\n{{Infobox b...,"(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
5,"'''''O come, O come, Emmanuel''''' is a [[Chri...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
6,{{Use mdy dates|date=July 2014}}\n{{Infobox ba...,"(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
7,{{Infobox baseball biography\n|name=Jake Gibbs...,"(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
8,{{about||the Led Zeppelin musician|John Paul J...,"(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
9,{{Infobox settlement\n| name ...,"(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."


##### Step 5: Normalizer

Cómo queremos usar un método que usa distancias (K-Means) suele ser aconsejable normalizar las variables de entrada para que su dimensión no altere en el resultado del algoritmo:

In [65]:
normalizer = Normalizer().setInputCol("idf").setOutputCol("features")

##### Step 6: k-means & tie it all together...

![](img/clustering.png)

In [66]:
from pyspark.ml import Pipeline
from pyspark.ml.clustering import KMeans

In [67]:
kmeans = KMeans().setFeaturesCol("features").setPredictionCol("prediction").setK(100).setSeed(1234)

In [68]:
pipeline = Pipeline(stages=[tokenizer, remover, hashing_tf, idf, normalizer, kmeans])

**CUIDADO** Este `fit` puede durar varios minutos:

In [69]:
model = pipeline.fit(wiki_df)

In [70]:
raw_predictions_df = model.transform(wiki_df).cache()

In [71]:
raw_predictions_df.count()

111495

En la variable `prediction` nos ha marcado en qué cluster a asignado cada artículo:

In [72]:
raw_predictions_df.select("prediction").limit(10).toPandas()

,prediction
0,31
1,5
2,31
3,32
4,50
5,31
6,50
7,50
8,29
9,39


¿Cuántos grupos hay?

In [73]:
raw_predictions_df.select("prediction").distinct().count()

100

In [74]:
raw_predictions_df.groupBy("prediction").count().orderBy(F.desc("count")).show(20)

+----------+-----+
|prediction|count|
+----------+-----+
|        31|37940|
|        33| 9575|
|         1| 5132|
|        77| 5125|
|        69| 4801|
|        85| 2965|
|        21| 2746|
|        29| 2746|
|        49| 2729|
|        62| 2631|
|         5| 2109|
|        32| 1909|
|        35| 1749|
|        39| 1727|
|        57| 1539|
|        42| 1467|
|         9| 1412|
|        30| 1243|
|        98| 1237|
|        88| 1100|
+----------+-----+
only showing top 20 rows



In [75]:
raw_predictions_df.filter(F.lower(F.col("title")).like("%hadoop%")).toPandas()

/opt/share/anaconda/lib/python3.11/site-packages/pyspark/sql/pandas/conversion.py:251: FutureWarning: Passing unit-less datetime64 dtype to .astype is deprecated and will raise in a future version. Pass 'datetime64[ns]' instead
  series = series.astype(t, copy=False)


,id,title,lastrev_pdt_time,revid,comment,contributorid,contributorusername,contributorip,text,words,noStopWords,hashingTF,idf,features,prediction
0,5919308,Apache Hadoop,2016-03-04 22:58:48,708371306,/* History */,NaN,None,71.84.15.41,{{multiple issues|\n{{advert|date=October 2013...,"[multiple, issues, advert, date, october, 2013...","[multiple, issues, advert, date, october, 2013...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 4.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 12.371279522029...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0086322528147...",31


In [76]:
raw_predictions_df.filter(F.lower(F.col("title")).like("%apache spark%")).toPandas()

/opt/share/anaconda/lib/python3.11/site-packages/pyspark/sql/pandas/conversion.py:251: FutureWarning: Passing unit-less datetime64 dtype to .astype is deprecated and will raise in a future version. Pass 'datetime64[ns]' instead
  series = series.astype(t, copy=False)


,id,title,lastrev_pdt_time,revid,comment,contributorid,contributorusername,contributorip,text,words,noStopWords,hashingTF,idf,features,prediction
0,42164234,Apache Spark,2016-03-03 14:13:40,708135330,relegate details to footnotes,196471,Qwertyus,None,{{Infobox Software\n| name =...,"[infobox, software, name, apache, spark, logo,...","[infobox, software, name, apache, spark, logo,...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",31


In [77]:
raw_predictions_df.filter("prediction = 24").select("title").limit(20).toPandas()

,title
0,Mick Doohan
1,Mark Smith (British racing driver)
2,Wayne Gardner
3,Germany's Next Topmodel (cycle 11)
4,Independent Spirit Award for Best Female Lead
5,Equipment of the Italian Army
6,List of Chief Ministers of Bihar
7,Hell's Kitchen (U.S. season 5)
8,UE Sant Andreu
9,Tube map


Parece que la temática de estos artículos es informática / tecnología

In [78]:
spark.stop()